# Will They Pay? — Model Training on Dummy Data (Dev)

> **Status:** Exploratory — superseded by `notebooks/production/train_model_prod.ipynb`.
> This was the initial proof-of-concept training run using fully synthetic data (all 250 persons and their features are fake). It established that the XGBoost pipeline worked end-to-end and that SMOTE helped with the class imbalance. Do not use it to regenerate model artifacts.

Loads the flat training table from `build_training_data.ipynb`, trains and compares baseline models (Logistic Regression, Random Forest, XGBoost), and evaluates performance.

In [ ]:
# Will They Pay? — Model Training & Evaluation
# Loads the flat training table from `build_training_data.ipynb`, trains and compares baseline models (Logistic Regression, Random Forest, XGBoost), evaluates performance, and batch-scores open invoices.

# **ML Workflow Steps:**
# 1. Load training data
# 2. Define features and target
# 3. Split data (80/20)
# 4. Train and compare models
# 5. Evaluate performance
# 6. Batch score open invoices

## Step 1: Load Training Data

In [ ]:
import pandas as pd
import numpy as np

training_df = pd.read_csv('../../data/processed/training_data.csv')

print(f"Loaded training data: {training_df.shape[0]} rows x {training_df.shape[1]} columns")
print(f"\nTarget distribution (paid_within_30_days):")
print(f"   Paid within 30d (1): {training_df['paid_within_30_days'].sum()} ({training_df['paid_within_30_days'].mean()*100:.1f}%)")
print(f"   Late/unpaid     (0): {(training_df['paid_within_30_days'] == 0).sum()} ({(1 - training_df['paid_within_30_days'].mean())*100:.1f}%)")
training_df.head()

## Step 2: Define Features and Target

In [ ]:
feature_columns = [
    # Invoice features
    'amount',
    'created_day_of_week',
    'created_day_of_month',
    'created_hour',
    'origin',
    'surchargingenabled',
    # Person features
    'is_guardian',
    'account_age_days',
    'tenure_months',
    'is_new_patient',
    # Derived behavioral features
    'average_days_to_pay',
    'appointment_reliability_score',
    # Census features
    'median_household_income',
    'poverty_rate_pct',
    'average_household_size',
    'bachelors_or_higher_pct',
    'unemployment_rate_pct',
]

X = training_df[feature_columns]
y = training_df['paid_within_30_days']

print(f"Features: {X.shape[1]} columns")
print(f"Target: {y.shape[0]} rows")
print(f"\nNull check:")
print(X.isnull().sum().to_string())

## Step 3: Split Data (Train / Test)

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"Train set: {X_train.shape[0]} rows")
print(f"Test set:  {X_test.shape[0]} rows")
print(f"\nTrain target distribution: {y_train.value_counts().to_dict()}")
print(f"Test target distribution:  {y_test.value_counts().to_dict()}")

## Step 4: Train and Compare Models
Trains three baseline models:
1. Logistic Regression — fast, interpretable
2. Random Forest — handles nonlinear interactions
3. XGBoost — often strongest performer

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

models = {
    'Logistic Regression': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', LogisticRegression(random_state=42, max_iter=1000, class_weight='balanced'))
    ]),
    'Random Forest': RandomForestClassifier(
        n_estimators=200, random_state=42, class_weight='balanced'
    ),
    'XGBoost': XGBClassifier(
        n_estimators=200, random_state=42, eval_metric='logloss',
        scale_pos_weight=(y_train == 0).sum() / (y_train == 1).sum()
    ),
}

trained_models = {}
for name, model in models.items():
    print(f"Training {name}...")
    model.fit(X_train, y_train)
    trained_models[name] = model
    print(f"   Done.")

print(f"\n\u2705 All {len(trained_models)} models trained.")

## Step 5: Evaluate Performance
Compares models by ROC-AUC, accuracy, precision/recall. Prioritizes precision on the "unlikely to pay" class per PRD.

In [ ]:
from sklearn.metrics import (
    accuracy_score, classification_report, roc_auc_score, confusion_matrix
)

results = []

for name, model in trained_models.items():
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]
    
    acc = accuracy_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_proba)
    
    results.append({'Model': name, 'Accuracy': acc, 'ROC-AUC': auc})
    
    print(f"\n{'='*60}")
    print(f" {name}")
    print(f"{'='*60}")
    print(f" Accuracy: {acc:.4f}")
    print(f" ROC-AUC:  {auc:.4f}")
    print(f"\n Classification Report:")
    print(classification_report(y_test, y_pred, target_names=['Unpaid', 'Paid']))
    print(f" Confusion Matrix:")
    print(confusion_matrix(y_test, y_pred))

# Summary comparison
results_df = pd.DataFrame(results).sort_values('ROC-AUC', ascending=False)
print(f"\n\n{'='*60}")
print(" MODEL COMPARISON (sorted by ROC-AUC)")
print(f"{'='*60}")
print(results_df.to_string(index=False))

best_model_name = results_df.iloc[0]['Model']
print(f"\n\u2705 Best model: {best_model_name} (AUC={results_df.iloc[0]['ROC-AUC']:.4f})")

## Step 5b: Feature Importance (Best Model)

In [ ]:
best_model = trained_models[best_model_name]

# Extract feature importances (works for tree-based models)
if hasattr(best_model, 'feature_importances_'):
    importances = best_model.feature_importances_
elif hasattr(best_model, 'named_steps'):
    # Pipeline (logistic regression) — use coefficients
    importances = np.abs(best_model.named_steps['clf'].coef_[0])
else:
    importances = None

if importances is not None:
    importance_df = pd.DataFrame({
        'Feature': feature_columns,
        'Importance': importances
    }).sort_values('Importance', ascending=False)
    
    print(f"Feature importance ({best_model_name}):")
    print(importance_df.to_string(index=False))

## Step 6: Save Model Artifact

In [ ]:
import joblib
import json
import os

artifacts_dir = '../../models'
os.makedirs(artifacts_dir, exist_ok=True)

# Save model
model_path = os.path.join(artifacts_dir, 'will_they_pay_v1.joblib')
joblib.dump(best_model, model_path)
print(f"\u2705 Saved model: {model_path}")

# Save feature columns (required by batch scorer per TPD)
features_path = os.path.join(artifacts_dir, 'feature_columns.json')
with open(features_path, 'w') as f:
    json.dump(feature_columns, f, indent=2)
print(f"\u2705 Saved feature columns: {features_path}")

## Step 7: Batch Score Open Invoices
Scores currently open (unpaid) invoices with the trained model. Outputs prediction + confidence + risk band per the TPD schema.

In [ ]:
# Filter to open invoices (UNPAID=6, PARTIALLY_PAID=5, PENDING=9, SCHEDULED=7)
open_statuses = [5, 6, 7, 9]
open_invoices = training_df[training_df['status'].isin(open_statuses)].copy()

print(f"Open invoices to score: {len(open_invoices)}")

# Extract features
X_open = open_invoices[feature_columns]

# Predict
open_invoices['will_pay_in_30'] = best_model.predict(X_open)
open_invoices['confidence_score'] = best_model.predict_proba(X_open)[:, 1]

# Derive risk band per TPD:
# >= 0.70 -> "high" (likely to pay)
# 0.40 - 0.69 -> "medium"
# < 0.40 -> "low" (unlikely to pay)
open_invoices['risk_band'] = pd.cut(
    open_invoices['confidence_score'],
    bins=[0, 0.40, 0.70, 1.0],
    labels=['low', 'medium', 'high'],
    include_lowest=True
)

print(f"\nRisk band distribution:")
print(open_invoices['risk_band'].value_counts().to_string())

print(f"\nConfidence score stats:")
print(f"   Mean:   {open_invoices['confidence_score'].mean():.3f}")
print(f"   Median: {open_invoices['confidence_score'].median():.3f}")
print(f"   Min:    {open_invoices['confidence_score'].min():.3f}")
print(f"   Max:    {open_invoices['confidence_score'].max():.3f}")

print(f"\nPreview:")
open_invoices[['personid', 'amount', 'status', 'will_pay_in_30', 'confidence_score', 'risk_band']].head(10)

## Step 8: Save Scored Predictions

In [ ]:
# Build prediction output matching TPD payment_predictions schema
from datetime import datetime

predictions_output = open_invoices[['personid', 'postal_code', 'will_pay_in_30', 'confidence_score', 'risk_band']].copy()
predictions_output['model_version'] = 'will_they_pay_v1'
predictions_output['scored_at'] = datetime.utcnow().isoformat()

output_path = '../../data/processed/scored_predictions.csv'
predictions_output.to_csv(output_path, index=False)
print(f"\u2705 Saved {len(predictions_output)} scored predictions to {output_path}")
predictions_output.head()